> **Experiment 3 — Safety agent, heterogeneous TMR.** Three different models (GPT-5-mini, Gemini 2.5 Flash, Qwen2.5-VL-3B) + majority vote, assessed with HIP-LLM and a pyAgrum Bayesian network.

# SMILES → ECHA Hazard Check with Three Agents

A minimal workshop version of the working V2 workflow:

1. **Research & Browser Agent** — SMILES → PubChem → ECHA, then cookie/legal-notice preparation.
2. **PixelRAG Agent** — uses the exact V2 `render_url(..., cdp_url=...)` capture path.
3. **Safety Agent** — uses the V2 vision-risk schema to read visible ECHA classifications.

The PixelRAG installation is pinned to the exact commit recorded in the successful V2 execution.


## 1. Install PixelRAG

In [ ]:
# Kaggle Internet must be enabled.
# Reproduce the exact PixelRAG revision recorded in the successful V2 run.
!python -m pip uninstall -y playwright >/dev/null 2>&1 || true

!rm -rf /kaggle/working/PixelRAG
!git clone https://github.com/StarTrail-org/PixelRAG.git /kaggle/working/PixelRAG
!git -C /kaggle/working/PixelRAG checkout --detach 7a3c3041bd37b1868d8573039ebdc60607ddb936

%pip -q install -U \
    pillow \
    websockets \
    requests \
    openai \
    langgraph \
    langchain-core \
    typing_extensions \
    zstandard

%pip -q install -e /kaggle/working/PixelRAG --no-deps

!PYTHONPATH=/kaggle/working/PixelRAG/render/src \
 python -m pixelrag_render.chrome install

!PYTHONPATH=/kaggle/working/PixelRAG/render/src \
 python -m pixelrag_render.chrome which

PIXELRAG_COMMIT = !git -C /kaggle/working/PixelRAG rev-parse HEAD
PIXELRAG_COMMIT = PIXELRAG_COMMIT[0].strip()
print("PixelRAG commit:", PIXELRAG_COMMIT)
assert PIXELRAG_COMMIT == "7a3c3041bd37b1868d8573039ebdc60607ddb936"
print("Pinned V2 PixelRAG installation completed")


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY").strip()
os.environ["OPENAI_API_KEY"] = user_secrets.get_secret("OPENAI_API_KEY").strip()

os.environ.pop("QWEN_API_KEY", None)
os.environ["QWEN_API_KEY"]  = "ollama"
os.environ["QWEN_BASE_URL"] = os.environ.get("QWEN_BASE_URL", "http://127.0.0.1:11434/v1") 
os.environ["QWEN_MODEL"]    = "qwen2.5vl:3b"

## 2. Load the Utility Module

In [ ]:
import importlib, sys
from pathlib import Path

# Looks for the utility module in the repo (../src) first, then in a Kaggle utility-script mount.
CANDIDATES = [
    Path.cwd().parent / "src",
    Path.cwd() / "src",
    Path("/kaggle/usr/lib/notebooks/burkaytuntrk/agentic_smiles_hazard_utils"),
]
UTILS_DIR = next((p for p in CANDIDATES if (p / "agentic_smiles_hazard_utils.py").exists()), None)
if UTILS_DIR is None:
    raise FileNotFoundError("agentic_smiles_hazard_utils.py not found; see README (Setup).")
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

sys.modules.pop("agentic_smiles_hazard_utils", None)
utils = importlib.import_module("agentic_smiles_hazard_utils")


In [ ]:
!pip install -q google-genai openai

import os, json, base64, copy, re
from google import genai
from google.genai import types
from openai import OpenAI
import requests
from io import BytesIO
from PIL import Image

GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-5-mini")
QWEN_MODEL   = os.environ.get("QWEN_MODEL",   "qwen2.5vl:3b")

_gclient = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
_oclient = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
_qclient = OpenAI(api_key=os.environ["QWEN_API_KEY"],
                  base_url=os.environ["QWEN_BASE_URL"],
                  timeout=900.0, max_retries=0)
# ---------- common helpers ----------
def _clean_schema(schema):
    schema = copy.deepcopy(schema)
    def strip(n):
        if isinstance(n, dict):
            n.pop("additionalProperties", None); n.pop("strict", None)
            for v in n.values(): strip(v)
        elif isinstance(n, list):
            for v in n: strip(v)
    strip(schema); return schema

def _extract_json(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    return json.loads(text)

def _to_parts(content):
    parts = []
    for it in content:
        if it.get("type") == "input_text":
            parts.append(types.Part.from_text(text=it["text"]))
        elif it.get("type") == "input_image":
            hdr, b64 = it["image_url"].split(",", 1)
            mime = hdr.split(";")[0].replace("data:", "") or "image/png"
            parts.append(types.Part.from_bytes(data=base64.b64decode(b64), mime_type=mime))
    return parts

def _to_chat(content):
    out = []
    for it in content:
        if it.get("type") == "input_text":
            out.append({"type": "text", "text": it["text"]})
        elif it.get("type") == "input_image":
            out.append({"type": "image_url",
                        "image_url": {"url": it["image_url"],
                                      "detail": it.get("detail", "high")}})
    return out

# ---------- MODULE 1: OpenAI ----------
def openai_structured_response(*, instructions, content, schema_name, schema, use_web_search=False):
    resp = _oclient.chat.completions.create(
        model=OPENAI_MODEL,
        max_completion_tokens=8192,
        messages=[{"role": "system", "content": instructions},
                  {"role": "user",   "content": _to_chat(content)}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": schema_name,
                                         "strict": True,
                                         "schema": schema}},
    )
    return _extract_json(resp.choices[0].message.content), []
    
# ---------- MODULE 2: Gemini ----------
def gemini_structured_response(*, instructions, content, schema_name, schema, use_web_search=False):
    cfg = types.GenerateContentConfig(
        system_instruction=instructions, temperature=0.0, max_output_tokens=16384,
        response_mime_type="application/json", response_schema=_clean_schema(schema))
    resp = _gclient.models.generate_content(model=GEMINI_MODEL,
                                            contents=_to_parts(content), config=cfg)
    return _extract_json(resp.text), []

# ---------- MODULE 3: Qwen ----------
def qwen_structured_response(*, instructions, content, schema_name, schema, use_web_search=False):
    base_url = os.environ.get("QWEN_BASE_URL", "http://127.0.0.1:11434").removesuffix("/v1").rstrip("/")
    api_url = f"{base_url}/api/chat"
    
    text_parts = []
    images = []
    
    for it in content:
        if it.get("type") == "input_text":
            text_parts.append(it["text"])
        elif it.get("type") == "input_image":
            raw_b64 = it["image_url"].split(",", 1)[-1].strip()
            
            # --- Make the image 800px in order to decrease token count ~1000 ---
            try:
                img_data = base64.b64decode(raw_b64)
                img = Image.open(io.BytesIO(img_data)).convert("RGB")
                img.thumbnail((800, 800), Image.Resampling.LANCZOS)
                
                buf = io.BytesIO()
                img.save(buf, format="JPEG", quality=75)
                resized_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
                images.append(resized_b64)
            except Exception:
                images.append(raw_b64)
            
    sys_msg = (
        instructions
        + "\n\n### CRITICAL EXTRACTION RULES ###"
          "\n- 'hazard_statement_code' MUST match the pattern 'Hxxx' (e.g., H225, H301, H319, H336, H420) or 'EUHxxx'."
          "\n- NEVER put pictogram labels (GHS01, GHS02, GHS07, GHS08, GHS09) into 'hazard_statement_code'."
          "\n- If you see 'GHS07', place it ONLY into the 'pictograms' array, NOT in 'classifications'."
          "\n- Any classification item where 'hazard_statement_code' starts with 'GHS' is STRICTLY FORBIDDEN."
          "\n\nReturn ONLY valid JSON matching this schema:\n"
        + json.dumps(_clean_schema(schema), ensure_ascii=False)
    )
    
    user_msg = {"role": "user", "content": "\n".join(text_parts)}
    if images:
        user_msg["images"] = [images[0]]

    payload = {
        "model": "qwen2.5vl:3b",
        "messages": [
            {"role": "system", "content": sys_msg},
            user_msg
        ],
        "stream": False,
        "format": "json",
        "options": {
            "temperature": 0.0,
            "num_ctx": 8192,
            "num_predict": 2048
        }
    }
    
    headers = {
        "ngrok-skip-browser-warning": "true",
        "Content-Type": "application/json"
    }
    
    res = requests.post(api_url, json=payload, headers=headers, timeout=900)
    
    if res.status_code != 200:
        raise RuntimeError(f"Ollama Error ({res.status_code}): {res.text}")
        
    res_json = res.json()
    raw_content = res_json.get("message", {}).get("content", "{}")
    parsed_json = _extract_json(raw_content)
    
    if isinstance(parsed_json, dict) and "classifications" in parsed_json:
        valid_cls = []
        for c in parsed_json.get("classifications", []):
            code = str(c.get("hazard_statement_code", "")).strip().upper()
            if re.match(r"^(H\d{3}[A-Z]*|EUH\d{3})$", code):
                valid_cls.append(c)
        parsed_json["classifications"] = valid_cls
        
    return parsed_json, []
# ---------- modul recording ----------
import agentic_smiles_hazard_utils as utils

MODULE_BACKENDS = {
    1: ("openai", OPENAI_MODEL, openai_structured_response),
    2: ("gemini", GEMINI_MODEL, gemini_structured_response),
    3: ("qwen",   QWEN_MODEL,   qwen_structured_response),
}
MODULE_MODELS = {m: f"{v[0]}/{v[1]}" for m, v in MODULE_BACKENDS.items()}
ACTIVE_MODULE = None

def set_active_module(m):
    global ACTIVE_MODULE
    utils.structured_response = MODULE_BACKENDS[m][2]
    ACTIVE_MODULE = m
    return MODULE_MODELS[m]

set_active_module(2)
print("modules:", MODULE_MODELS)

## 3. PixelRAG Browser Self-Test

In [ ]:
from agentic_smiles_hazard_utils import pixelrag_self_test

pixelrag_self_test()


## 5. Shared State

In [ ]:
import typing

class AgentState(typing.TypedDict, total=False):
    smiles: str
    identity: dict[str, typing.Any]
    echa_resolution: dict[str, typing.Any]
    browser: dict[str, typing.Any]
    browser_status: str
    capture: dict[str, typing.Any]
    capture_status: str
    capture_attempts: int
    risk: dict[str, typing.Any]
    final_record: dict[str, typing.Any]
    text_report: str
    workflow_succeeded: bool
    error: str


## 6. Agent 1 — Research & Browser Preparation Agent

This agent resolves the SMILES and exact ECHA URL, then prepares the shared browser profile. It dismisses the cookie banner when present, accepts the ECHA legal notice, and verifies that `Industry classifications` / `CLP Submission` are rendered before PixelRAG runs.


In [ ]:
def research_agent(state: AgentState) -> AgentState:
    from agentic_smiles_hazard_utils import (
        browser_session_agent,
        lookup_echa,
        lookup_identity,
    )

    identity = lookup_identity(state["smiles"])
    if identity.get("status") != "resolved":
        return {
            "identity": identity,
            "browser_status": "failed",
            "workflow_succeeded": False,
            "error": identity.get("error", "Identity lookup failed."),
        }

    resolution = lookup_echa(identity)
    if resolution.get("status") != "found":
        return {
            "identity": identity,
            "echa_resolution": resolution,
            "browser_status": "failed",
            "workflow_succeeded": False,
            "error": resolution.get("notes", "ECHA record not found."),
        }

    browser_update = browser_session_agent({"echa_resolution": resolution})

    return {
        "identity": identity,
        "echa_resolution": resolution,
        "browser": browser_update.get("browser", {}),
        "browser_status": browser_update.get("browser_status", "failed"),
    }


## 7. Agent 2 — PixelRAG Agent

This deliberately calls the **original V2 capture function unchanged**. It uses `PixelRAG.render_url()` with the prepared Chrome CDP browser—the same path that produced the working Clozapine hazard screenshot in V2.


In [ ]:
def pixelrag_agent(state: AgentState) -> AgentState:
    from agentic_smiles_hazard_utils import pixelrag_capture_agent

    update = pixelrag_capture_agent(state)

    return {
        "capture": update.get("capture", {}),
        "capture_status": update.get("capture_status", "failed"),
        "capture_attempts": update.get(
            "capture_attempts",
            state.get("capture_attempts", 0),
        ),
    }


## 8. Agent 3 — Safety Agent

The hazard-reading part below is copied from the working V2 **Vision Risk Agent**: the same structured schema, evidence rules and image-reading prompt are retained. This is the important part that previously extracted three visible Clozapine classifications from the PixelRAG tile.


In [ ]:
RISK_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "evidence_status": {
            "type": "string",
            "enum": ["sufficient", "partial", "insufficient"],
        },
        "substance_match": {
            "type": "string",
            "enum": ["exact", "probable", "not_verified"],
        },
        "substance_name": {"type": "string"},
        "cas": {"type": "string"},
        "ec_number": {"type": "string"},
        "submission": {"type": "string"},
        "notification_share": {"type": "string"},
        "status": {"type": "string"},
        "last_updated": {"type": "string"},
        "signal_word": {"type": "string"},
        "classifications": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "hazard_class_category": {"type": "string"},
                    "hazard_statement_code": {
                        "type": "string",
                        "description": "The exact hazard code starting strictly with 'H' or 'EUH' (e.g. H225, H301, H319, H336, H370). NEVER put GHS codes here."
                    },
                    "hazard_statement_text": {"type": "string"},
                    "reason_for_classification": {"type": "string"},
                    "affected_organs_routes": {"type": "string"},
                },
                "required": [
                    "hazard_class_category",
                    "hazard_statement_code",
                    "hazard_statement_text",
                    "reason_for_classification",
                    "affected_organs_routes",
                ],
            },
        },
        "pictograms": {
            "type": "array",
            "items": {"type": "string"},
        },
        "key_risks": {
            "type": "array",
            "items": {"type": "string"},
        },
        "visual_evidence_notes": {"type": "string"},
    },
    "required": [
        "evidence_status",
        "substance_match",
        "substance_name",
        "cas",
        "ec_number",
        "submission",
        "notification_share",
        "status",
        "last_updated",
        "signal_word",
        "classifications",
        "pictograms",
        "key_risks",
        "visual_evidence_notes",
    ],
}

def safety_agent(state: AgentState) -> AgentState:
    # Hazard extraction below is the V2 Vision Risk Agent logic.
    import json
    from pathlib import Path
    from agentic_smiles_hazard_utils import encode_image, structured_response

    identity = state["identity"]
    capture = state["capture"]

    tile_paths = [Path(value) for value in capture.get("tile_paths", [])]
    if not tile_paths:
        return {
            "workflow_succeeded": False,
            "error": "The PixelRAG capture contained no usable tiles.",
        }

    content = [
        {
            "type": "input_text",
            "text": json.dumps(
                {
                    "task": (
                        "Read the supplied PixelRAG screenshots of an ECHA CHEM page "
                        "and extract only risk information visibly supported by them."
                    ),
                    "expected_identity": {
                        "name": identity.get("name", ""),
                        "cas": identity.get("cas", ""),
                        "ec_number": state.get("echa_resolution", {}).get(
                            "matched_ec_number", ""
                        ),
                    },
                    "rules": [
                        "Do not infer hidden fields.",
                        "Do not use general chemical knowledge.",
                        "Mark evidence partial or insufficient when the relevant panel is absent.",
                        "Require visible identity evidence before declaring an exact match.",
                        "Transcribe percentages, dates, codes, labels, and pictograms carefully.",
                        "Differentiate pictograms (GHSxx) from hazard codes (Hxxx); never place GHS codes into classifications.",
                    ],
                },
                ensure_ascii=False,
            ),
        }
    ]

    for path in tile_paths:
        content.append(
            {
                "type": "input_image",
                "image_url": encode_image(path),
                "detail": "high",
            }
        )

    risk, _ = structured_response(
        instructions=(
            "You are the Vision Risk Agent. You read PixelRAG screenshot tiles "
            "captured from ECHA CHEM. Extract only visually supported content. "
            "Be conservative and exact."
        ),
        content=content,
        schema_name="pixelrag_echa_risk",
        schema=RISK_SCHEMA,
        use_web_search=False,
    )

    resolution = state.get("echa_resolution", {})
    record = {
        "smiles": state["smiles"],
        "identity": identity,
        "echa_url": resolution.get("industry_url") or resolution.get("substance_url", ""),
        "pixelrag_tiles": [str(path) for path in tile_paths],
        "risk": risk,
    }

    lines = [
        f"Substance: {risk.get('substance_name') or identity.get('name', '')}",
        f"CAS: {risk.get('cas') or identity.get('cas', '')}",
        f"Signal word: {risk.get('signal_word', '')}",
        "",
        "ECHA classifications:",
    ]

    for row in risk.get("classifications", []):
        parts = [
            row.get("hazard_class_category", ""),
            row.get("hazard_statement_code", ""),
            row.get("hazard_statement_text", ""),
        ]
        lines.append("- " + " | ".join(part for part in parts if part))

    if risk.get("pictograms"):
        lines.extend(["", "Pictograms: " + ", ".join(risk["pictograms"])])

    return {
        "risk": risk,
        "final_record": record,
        "text_report": "\n".join(lines),
        "workflow_succeeded": risk.get("evidence_status") != "insufficient",
        "error": (
            ""
            if risk.get("evidence_status") != "insufficient"
            else "The screenshots did not contain sufficient visible ECHA hazard evidence."
        ),
    }
#############################################################################

In [ ]:
import os
import re
from pathlib import Path

htmls = sorted(Path(os.environ.get("ECHA_HTML_DIR", "/kaggle/input")).rglob("*.html"))
print("HTML files:")
for h in htmls:
    print("  ", h)

def ground_truth_from_html(path):
    raw = Path(path).read_text(encoding="utf-8", errors="ignore")
    t = re.sub(r"<script.*?</script>|<style.*?</style>", " ", raw, flags=re.S | re.I)
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\s+", " ", t)
    codes = set(re.findall(r"\b(H\d{3}[A-Z]*|EUH\d{3})\b", t))
    return {"codes": codes}

print("\n--- GROUND TRUTH ---")
for h in htmls:
    gt = ground_truth_from_html(h)
    print(f"\n{h.name}")
    print("  H codes   :", sorted(gt["codes"]))

In [ ]:
import subprocess, tempfile, os, time
from pathlib import Path
import agentic_smiles_hazard_utils as utils
from pixelrag_render.chrome import find_chrome
from pixelrag_render.backends.cdp import BROWSER_ARGS

port = utils.find_free_port(9444)
CDP_URL = f"http://127.0.0.1:{port}"
profile = Path(tempfile.mkdtemp(prefix="render_"))

proc = subprocess.Popen(
    [
        find_chrome(auto_install=False),
        f"--remote-debugging-port={port}",
        "--headless=new",
        f"--user-data-dir={profile}",
        *BROWSER_ARGS,
        "--allow-file-access-from-files",
        "--window-size=1600,1100",
        "about:blank",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=os.environ.copy(),
)

for _ in range(60):
    try:
        utils.cdp_json(f"{CDP_URL}/json/version")
        break
    except Exception:
        time.sleep(0.5)

print("Chrome ready:", CDP_URL, "| pid:", proc.pid)

In [ ]:
import os
# === TMR raw collection===
import shutil, re, time, json, csv, os
from pathlib import Path
import pandas as pd
import agentic_smiles_hazard_utils as utils

RUN_COUNT    = 10 #run count for each molecule
N_MODULE   = 3 #how many module for the NMR
MAX_ATTEMPTS = 3 #in case of an infra error, this is the total retry amount
RETRY_BACKOFF = (5, 5) # how many seconds system wait for the infra retry attempt
RAW_CSV      = Path("/kaggle/working/safety_eval_raw_3models.csv")
FORCE_RERUN  = False  # if it is set true, it will override the existing output

INFRA_MARKERS = (
    "401", "403", "PERMISSION_DENIED", "UNAUTHENTICATED", "API key",
    "API_KEY_INVALID", "503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED",
    "quota", "deadline", "timeout", "Connection", "ServerError",
)

def classify_error(e):
    """infra = not the models fault -> retry"""
    msg = f"{type(e).__name__}: {e}"
    if any(m.lower() in msg.lower() for m in INFRA_MARKERS):
        return "infra"
    if isinstance(e, json.JSONDecodeError):
        return "infra"
    return "model"

FIELDS = ["substance", "source_file", "run", "replica", "model", "status", "attempts",
          "error", "time_s", "codes", "signal_word", "evidence_status",
          "substance_match", "n_classifications", "risk_json"]

def append_row(row):
    new = not RAW_CSV.exists()
    with open(RAW_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)

def _find_previous():
    if RAW_CSV.exists():
        return RAW_CSV
    cands = sorted(Path("/kaggle/input").rglob(RAW_CSV.name))
    return cands[0] if cands else None

if FORCE_RERUN and RAW_CSV.exists():
    RAW_CSV.unlink()
    print("FORCE_RERUN: existing CSV deleted")

prev = _find_previous()
if prev is not None and prev != RAW_CSV and not FORCE_RERUN:
    shutil.copy(prev, RAW_CSV)
    print("previous output found:", prev)

done = set()
if RAW_CSV.exists():
    _old = pd.read_csv(RAW_CSV, keep_default_na=False)
    missing = set(FIELDS) - set(_old.columns)
    if missing:
        raise RuntimeError(
            f"current CSV is faulty (missing collon: {sorted(missing)}). "
            f"delete or move: {RAW_CSV}"
        )
        
    done = {(str(r.substance), int(r.run), int(r.replica)) for r in _old.itertuples()}
    print(f"CSV current call: {len(done)}")


srcs = sorted(Path(os.environ.get("ECHA_HTML_DIR", "/kaggle/input")).rglob("*.html"))
plan = []
for src in srcs:
    label = src.name.split(" 100.")[0].removesuffix(".html").strip()
    todo = [(r, rep) for r in range(1, RUN_COUNT + 1)
                     for rep in range(1, N_MODULE + 1)
                     if (label, r, rep) not in done]
    if todo:
        plan.append((src, label, todo))

if not plan:
    print(f"✅ All the outputs ready({len(done)}). "
          f"0 API request sent, not rendered.")
else:
    n_calls = sum(len(t) for _, _, t in plan)
    print(f"⚠️  {len(plan)} molecule, {n_calls} new API call will be made\n")

    for src, label, todo in plan:
        work = Path("/kaggle/working/eval") / re.sub(r"\W+", "_", label)
        if work.exists():
            shutil.rmtree(work)
        work.mkdir(parents=True, exist_ok=True)
        dst = work / "page.html"
        shutil.copy(src, dst)
        fdir = src.parent / (src.stem + "_files")
        if fdir.exists():
            shutil.copytree(fdir, work / fdir.name, dirs_exist_ok=True)
        tdir = work / "tiles"
        tdir.mkdir(exist_ok=True)

        try:
            tiles = utils.collect_tiles(utils.render_tiles(f"file://{dst}", tdir, CDP_URL), {})
        except Exception as e:
            print(f"{label}: render error ({type(e).__name__}) -> molecule assigned")
            continue

        state = {
            "smiles": "",
            "identity": {"name": label, "cas": ""},
            "echa_resolution": {"status": "found", "matched_name": label,
                                "matched_cas": "", "matched_ec_number": ""},
            "capture": {"tile_paths": [str(t) for t in tiles]},
        }

        for run_id, rep in todo:
            model_name = set_active_module(rep)     # 1=openai 2=gemini 3=qwen
            base = {"substance": label, "source_file": src.name,
                    "run": run_id, "replica": rep, "model": model_name}
            risk, last_err, attempts = None, "", 0
            t0 = time.time()

            for attempt in range(1, MAX_ATTEMPTS + 1):
                attempts = attempt
                try:
                    risk = safety_agent(state).get("risk", {})
                    last_err = ""
                    break
                except Exception as e:
                    kind = classify_error(e)
                    last_err = f"{type(e).__name__}: {str(e)[:200]}"
                    if kind == "model":
                        risk = None
                        break
                    if attempt < MAX_ATTEMPTS:
                        wait = RETRY_BACKOFF[min(attempt - 1, len(RETRY_BACKOFF) - 1)]
                        print(f"{label:24} run {run_id} rep {rep}  "
                              f"infra ({type(e).__name__}) -> retry {attempt}/{MAX_ATTEMPTS - 1} "
                              f"({wait}s waiting)")
                        time.sleep(wait)
                    else:
                        risk = None

            elapsed = round(time.time() - t0, 1)

            if risk is None:
                kind = classify_error(Exception(last_err))
                status = "infra_error" if kind == "infra" else "model_error"
                append_row({**base, "status": status, "attempts": attempts,
                            "error": last_err, "time_s": elapsed,
                            "codes": "", "signal_word": "", "evidence_status": "",
                            "substance_match": "", "n_classifications": "",
                            "risk_json": ""})
                mark = "-1 (infra, 3 trial failed)" if status == "infra_error" else "0 (model error)"
                print(f"{label:24} run {run_id} module {rep}  -> {mark}")
                continue

            cls = risk.get("classifications", []) or []
            codes = sorted({c.get("hazard_statement_code", "") for c in cls} - {""})
            append_row({**base, "status": "ok", "attempts": attempts, "error": "",
                        "time_s": elapsed,
                        "codes": "|".join(codes),
                        "signal_word": risk.get("signal_word", ""),
                        "evidence_status": risk.get("evidence_status", ""),
                        "substance_match": risk.get("substance_match", ""),
                        "n_classifications": len(cls),
                        "risk_json": json.dumps(risk, ensure_ascii=False)})
            print(f"{label:24} run {run_id} module {rep}  codes={codes}  ({elapsed}s)")

raw = pd.read_csv(RAW_CSV, keep_default_na=False) if RAW_CSV.exists() else pd.DataFrame()
display(raw.drop(columns=["risk_json"], errors="ignore"))
if len(raw):
    print(f"\ntotal: {len(raw)}  |  ok: {(raw['status']=='ok').sum()}  "
          f"|  infra: {(raw['status']=='infra_error').sum()}  "
          f"|  model: {(raw['status']=='model_error').sum()}")

In [ ]:
import os
# === Scoring + TMR voting (3 different model) ===
import json
import pandas as pd
from pathlib import Path
from collections import Counter

RAW_CSV = Path("/kaggle/working/safety_eval_raw_3models.csv")
TMR_CSV = Path("/kaggle/working/tmr_eval.csv")

raw = pd.read_csv(RAW_CSV, keep_default_na=False)

src_map = {p.name: p for p in Path(os.environ.get("ECHA_HTML_DIR", "/kaggle/input")).rglob("*.html")}
gt_cache = {}
def gt_for(source_file):
    if source_file not in gt_cache:
        p = src_map.get(source_file)
        if p is None:
            raise FileNotFoundError(f"HTML has not been found: {source_file}")
        gt_cache[source_file] = ground_truth_from_html(p)["codes"]
    return gt_cache[source_file]

def as_set(s):
    return set(x for x in str(s).split("|") if x)

def score_row(r):
    if r.status == "infra_error":
        return -1
    if r.status == "model_error":
        return 0
    return int(as_set(r.codes) == gt_for(r.source_file))

raw["expected"] = ["|".join(sorted(gt_for(f))) for f in raw["source_file"]]
raw["hazard_type"] = raw["expected"].apply(
    lambda e: "null" if not e else ("single" if len(as_set(e)) == 1 else "multi"))
raw["call_success"] = [score_row(r) for r in raw.itertuples()]

print("=== per-model (per-call) ===")
per = raw[raw["call_success"] >= 0]
print(per.groupby(["replica", "model"])["call_success"]
         .agg(["count", "sum", "mean"]).round(3).to_string())
print(f"infra (-1) row: {(raw['call_success'] == -1).sum()}")

def majority_vote(code_sets):
    thr = len(code_sets) // 2 + 1
    c = Counter()
    for s in code_sets:
        c.update(s)
    return {code for code, cnt in c.items() if cnt >= thr}

rows = []
for (label, run_id), g in raw.groupby(["substance", "run"], sort=False):
    exp = as_set(g["expected"].iloc[0])
    hazard_type = g["hazard_type"].iloc[0]
    ok = g[g["status"] == "ok"]

    # WITHOUT TMR
    for rep, gm in g.groupby("replica"):
        mname = gm["model"].iloc[0]
        if gm["status"].iloc[0] == "ok":
            got = as_set(gm["codes"].iloc[0]); s_no = int(got == exp)
        else:
            got = set()
            s_no = -1 if gm["status"].iloc[0] == "infra_error" else 0
        rows.append({"system": f"no_tmr_{mname.split('/')[0]}", "module": int(rep),
                     "model": mname, "substance": label, "run": run_id,
                     "stratum": hazard_type, "success": s_no, "n_valid": 1,
                     "expected": "|".join(sorted(exp)), "got": "|".join(sorted(got)),
                     "unanimous": ""})

    # WITH TMR -> 3 differnt model consensus
    sets = [as_set(v) for v in ok["codes"]]
    if len(sets) >= 2:
        voted = majority_vote(sets)
        rows.append({"system": "tmr", "module": 0, "model": "vote(3 models)",
                     "substance": label, "run": run_id, "stratum": hazard_type,
                     "success": int(voted == exp), "n_valid": len(sets),
                     "expected": "|".join(sorted(exp)), "got": "|".join(sorted(voted)),
                     "unanimous": int(all(s == sets[0] for s in sets))})
    else:
        rows.append({"system": "tmr", "module": 0, "model": "vote(3 models)",
                     "substance": label, "run": run_id, "stratum": hazard_type,
                     "success": -1, "n_valid": len(sets),
                     "expected": "|".join(sorted(exp)), "got": "", "unanimous": ""})

tmr = pd.DataFrame(rows)
tmr.to_csv(TMR_CSV, index=False)
display(tmr)

sc = tmr[tmr["success"] >= 0]
print("\n=== success rate ===")
print(sc.groupby("system")["success"].agg(["count", "sum", "mean"]).round(3).to_string())
print("\n=== stratum based ===")
print(sc.groupby(["system", "stratum"])["success"]
        .agg(["count", "sum", "mean"]).round(3).to_string())

una = tmr[(tmr["system"] == "tmr") & (tmr["unanimous"] != "")]
if len(una):
    print(f"\n3 model consensus: {una['unanimous'].astype(int).mean():.1%} "
          "(close to 0 means the models disagree strongly)")

## 12. Clean Up the Browser

In [ ]:
from agentic_smiles_hazard_utils import cleanup_browser

cleanup_browser(globals().get("state", {}).get("browser", {}))

try:
    proc.terminate(); proc.wait(timeout=10)
    print("Chrome turned off, pid:", proc.pid)
except Exception as e:
    print("Chrome :", type(e).__name__, e)

## 13. HIP LLM

In [ ]:
!pip -q install git+https://github.com/koo-ec/HIP_LLM.git
import pandas as pd
from HIPLLM import OperationalFailureProb, quick_inference_settings

tmr = pd.read_csv("/kaggle/working/tmr_eval.csv")
tmr = tmr[tmr["success"] >= 0].copy()          # exclude -1 (infra)

op = {"single": 0.2, "multi": 0.8}
op = {k: v / sum(op.values()) for k, v in op.items()}
print("operational profile (weights applied last):", op)
tmr = tmr[tmr["stratum"].isin(op.keys())]

hip = {}
for system, g in tmr.groupby("system"):
    est = OperationalFailureProb(
        profile=op,
        settings=quick_inference_settings(samples=1500, configurations=48),
    )
    res = est.fit(outcomes=g["success"].astype(int).tolist(), strata=g["stratum"].tolist())
    hip[system] = res
    print(f"\n########## {system.upper()} (n={len(g)}) ##########")
    print(res.summary())
    display(res.to_df())

In [ ]:
# === HIP-LLM Bayesian networks===
get_ipython().system("pip -q install pyagrum")

import itertools
import re
from IPython.display import HTML, display
import numpy as np
import pandas as pd
from HIPLLM import OperationalFailureProb, quick_inference_settings

try:
    import pyagrum as gum
    import pyagrum.lib.notebook as gnb
except ImportError:
    import pyAgrum as gum
    import pyAgrum.lib.notebook as gnb

OK, FAIL = "ok", "fail"
MODULES = [1, 2, 3]
STRATA = list(op.keys())
P_VOTER_FAIL = 1e-3
HIP_SETTINGS = dict(samples=1200, configurations=32)

def clean_var_name(name):
    short_name = name.split("/")[-1]
    return re.sub(r"[^a-zA-Z0-9_]", "_", short_name)

MOD_NAMES = {m: clean_var_name(MODULE_MODELS[m]) for m in MODULES}
PAIRS = list(itertools.combinations(MODULES, 2))

# ---------- 1. Per Module HIP Failure Bound ----------
per = raw[(raw["call_success"] >= 0) & (raw["hazard_type"].isin(STRATA))]

q_lo, q_hi = {}, {}
for m in MODULES:
    g = per[per["replica"] == m]
    if g.empty:
        raise ValueError(f"module {m} ({MODULE_MODELS[m]}) has no scored rows")
    res = OperationalFailureProb(
        profile=op,
        settings=quick_inference_settings(**HIP_SETTINGS),
    ).fit(
        outcomes=g["call_success"].astype(int).tolist(),
        strata=g["hazard_type"].astype(str).tolist(),
    )
    df = res.to_df().set_index("stratum")
    for s in STRATA:
        q_lo[(s, m)] = 1.0 - float(df.loc[s, "posterior_mean_reliability_upper"])
        q_hi[(s, m)] = 1.0 - float(df.loc[s, "posterior_mean_reliability_lower"])

q_table = pd.DataFrame(
    [
        {
            "stratum": s,
            "module_node": MOD_NAMES[m],
            "raw_model": MODULE_MODELS[m],
            "q_lo": q_lo[(s, m)],
            "q_hi": q_hi[(s, m)],
        }
        for s in STRATA
        for m in MODULES
    ]
)
print("=== per-module HIP failure bounds ===")
display(q_table.round(4))

# ---------- 2. pyAgrum Bayes network Function ----------
def build_bn(q, name):
    """q: {module: P(fail)} - Model isimleriyle pyAgrum BN kurar."""
    bn = gum.BayesNet(name)

    def add(n, d):
        bn.add(gum.LabelizedVariable(n, d, [OK, FAIL]))

    for m in MODULES:
        add(MOD_NAMES[m], MODULE_MODELS[m])
    for a, b in PAIRS:
        node_and = f"AND_{MOD_NAMES[a]}_{MOD_NAMES[b]}"
        add(node_and, f"{MOD_NAMES[a]} & {MOD_NAMES[b]}")
        
    add("OR_2oo3", "2-out-of-3 logic")
    add("Voter", "Voting mechanism")
    add("Sys_TMR", "TMR System Result")

    for a, b in PAIRS:
        node_and = f"AND_{MOD_NAMES[a]}_{MOD_NAMES[b]}"
        bn.addArc(MOD_NAMES[a], node_and)
        bn.addArc(MOD_NAMES[b], node_and)
        bn.addArc(node_and, "OR_2oo3")
    bn.addArc("OR_2oo3", "Sys_TMR")
    bn.addArc("Voter", "Sys_TMR")

    for m in MODULES:
        bn.cpt(MOD_NAMES[m]).fillWith([1.0 - q[m], q[m]])
    bn.cpt("Voter").fillWith([1.0 - P_VOTER_FAIL, P_VOTER_FAIL])

    for a, b in PAIRS:
        node_and = f"AND_{MOD_NAMES[a]}_{MOD_NAMES[b]}"
        for va, vb in itertools.product([OK, FAIL], repeat=2):
            good = (va == OK and vb == OK)
            bn.cpt(node_and)[{MOD_NAMES[a]: va, MOD_NAMES[b]: vb}] = (
                [1.0, 0.0] if good else [0.0, 1.0]
            )

    for combo in itertools.product([OK, FAIL], repeat=len(PAIRS)):
        ev = {f"AND_{MOD_NAMES[a]}_{MOD_NAMES[b]}": v for (a, b), v in zip(PAIRS, combo)}
        bn.cpt("OR_2oo3")[ev] = (
            [1.0, 0.0] if any(v == OK for v in combo) else [0.0, 1.0]
        )

    for vo, vv in itertools.product([OK, FAIL], repeat=2):
        good = (vo == OK and vv == OK)
        bn.cpt("Sys_TMR")[{"OR_2oo3": vo, "Voter": vv}] = (
            [1.0, 0.0] if good else [0.0, 1.0]
        )

    return bn


def p_fail(bn, node):
    ie = gum.LazyPropagation(bn)
    ie.makeInference()
    return float(ie.posterior(node)[{node: FAIL}])


BNS = {
    s: (
        build_bn({m: q_lo[(s, m)] for m in MODULES}, f"{s}_Optimistic"),
        build_bn({m: q_hi[(s, m)] for m in MODULES}, f"{s}_Pessimistic"),
    )
    for s in STRATA
}

# ---------- 3. pyAgrum Native ----------
for s in STRATA:
    lo_bn, hi_bn = BNS[s]
    w = op[s]

    opt_svg = gnb.getInference(lo_bn, size="8")
    pess_svg = gnb.getInference(hi_bn, size="8")

    layout_html = f"""
    <div style="border-bottom: 2px solid #ddd; margin-bottom: 25px; padding-bottom: 15px;">
        <h3 style="margin-bottom: 8px; font-family: sans-serif;">
            Stratum: <span style="color:#2980b9;">{s}</span> 
            <span style="font-size: 0.85em; color: #7f8c8d;">(Weight: {w:.2f})</span>
        </h3>
        <div style="display: flex; gap: 20px; justify-content: flex-start; align-items: flex-start; overflow-x: auto;">
            <div style="flex: 1; min-width: 380px;">
                <h4 style="text-align: center; color: #27ae60; margin: 5px 0; font-family: sans-serif;">Optimistic (Lower Bound)</h4>
                <div>{opt_svg}</div>
            </div>
            <div style="flex: 1; min-width: 380px;">
                <h4 style="text-align: center; color: #c0392b; margin: 5px 0; font-family: sans-serif;">Pessimistic (Upper Bound)</h4>
                <div>{pess_svg}</div>
            </div>
        </div>
    </div>
    """
    display(HTML(layout_html))

# ---------- 4. numerical summary table ----------
rows = []
for s in STRATA:
    lo_bn, hi_bn = BNS[s]
    r = {"stratum": s, "weight": op[s]}
    for m in MODULES:
        node_name = MOD_NAMES[m]
        r[f"noTMR_{node_name}_lo"] = p_fail(lo_bn, node_name)
        r[f"noTMR_{node_name}_hi"] = p_fail(hi_bn, node_name)
    r["TMR_lo"] = p_fail(lo_bn, "Sys_TMR")
    r["TMR_hi"] = p_fail(hi_bn, "Sys_TMR")
    rows.append(r)

bn_table = pd.DataFrame(rows).set_index("stratum")
total = {
    c: float((bn_table[c] * bn_table["weight"]).sum())
    for c in bn_table.columns
    if c != "weight"
}
total["weight"] = 1.0
bn_table.loc["total (weighted last)"] = total

print("\n=== Summary Table ===")
display(bn_table.round(4))

In [ ]:
#graph downloader
import os
import shutil
from IPython.display import FileLink, display
OUTPUT_DIR = "/kaggle/working/bayesian_graphs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
for s in STRATA:
    lo_bn, hi_bn = BNS[s]
    opt_svg_str = gnb.getInference(lo_bn, size="8")
    pess_svg_str = gnb.getInference(hi_bn, size="8")
    with open(f"{OUTPUT_DIR}/{s}_optimistic.svg", "w", encoding="utf-8") as f:
        f.write(opt_svg_str)
        
    with open(f"{OUTPUT_DIR}/{s}_pessimistic.svg", "w", encoding="utf-8") as f:
        f.write(pess_svg_str)

print(f"✅ images '{OUTPUT_DIR}'saved.")
shutil.make_archive("/kaggle/working/bayesian_graphs", 'zip', OUTPUT_DIR)
print("\nto download, click:")
display(FileLink("bayesian_graphs.zip"))